# 11장 오차 역전파 — 연쇄 법칙, 계산 그래프, 넘파이 역전파, 자동 미분, 기울기 소실
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 연쇄 법칙을 숫자로 확인하기
2. 계산 그래프로 선형 회귀의 기울기 구하기
3. 2-2-1 신경망의 역전파를 넘파이로 구현하고 수치 미분으로 검산하기
4. 역전파로 XOR 학습하기
5. 텐서플로의 자동 미분으로 같은 기울기 구하기
6. 층이 깊을 때의 기울기 소실을 시그모이드와 ReLU로 비교하기

위에서부터 차례로 실행하세요.

## 1. 숫자로 따라가는 연쇄 법칙
$y = 3x^2 + 1$을 $u = x^2$, $y = 3u + 1$의 두 단계로 계산합니다. $x$가 0.001 변하면 $u$는 4배, $y$는 다시 3배 변해 전체 기울기는 $4 \times 3 = 12$입니다(이북 그림 11.2).

In [ ]:
def f(x):
    u = x ** 2              # 첫 단계: u = x²
    return 3 * u + 1        # 둘째 단계: y = 3u + 1

h = 0.001
print("y(2)       =", f(2))
print("y(2.001)   =", f(2 + h))
print("변화의 비율 =", (f(2 + h) - f(2)) / h)

## 2. 계산 그래프
8장의 선형 회귀에서 기록 하나($x = 2$, $y = 7$, $w = b = 1$)의 제곱 오차를 곱셈, 덧셈, 뺄셈, 제곱 노드로 나누어 순전파하고, 오른쪽에서 왼쪽으로 기울기를 전합니다(이북 그림 11.3). 덧셈 노드는 기울기를 그대로, 곱셈 노드는 상대편 값을 곱해서, 함수 노드는 도함수 값을 곱해서 전합니다.

In [ ]:
x, y = 2, 7                 # 입력과 정답
w, b = 1, 1                 # 지금의 가중치와 편향
# 순전파: 왼쪽에서 오른쪽으로 값 계산
p = w * x                   # 곱셈 노드
y_hat = p + b               # 덧셈 노드
e = y_hat - y               # 뺄셈 노드: 오차
L = e ** 2                  # 제곱 노드: 손실
print("순전파:", p, y_hat, e, L)
# 역전파: 오른쪽에서 왼쪽으로 기울기 계산
dL_de = 2 * e               # 제곱 노드: 도함수 2e를 곱한다
dL_dyhat = dL_de * 1        # 뺄셈 노드(e = ŷ - y): ŷ 쪽으로는 그대로
dL_db = dL_dyhat * 1        # 덧셈 노드: 양쪽에 그대로 전한다
dL_dp = dL_dyhat * 1
dL_dw = dL_dp * x           # 곱셈 노드: 상대편 입력 x를 곱한다
print("역전파:", dL_de, dL_dyhat, dL_db, dL_dw)

구한 기울기로 경사 하강법을 한 걸음 진행합니다.

In [ ]:
lr = 0.05                   # 학습률
w, b = w - lr * dL_dw, b - lr * dL_db
print("한 걸음 뒤:", w, b, "손실", (w * x + b - y) ** 2)

## 3. 2-2-1 신경망의 역전파
이북 그림 11.5의 신경망에 입력 (1, 0), 정답 1을 넣습니다. 주석의 번호는 이북 11.4절의 역전파 단계입니다.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

P = {"w1": 0.6, "w2": 0.4, "b1": 0.5,       # 은닉 뉴런 h1
     "w3": -0.4, "w4": 0.3, "b2": -0.7,     # 은닉 뉴런 h2
     "w5": 1.0, "w6": -1.0, "b3": -0.5}     # 출력 뉴런

def forward(P, x1, x2):
    h1 = sigmoid(P["w1"] * x1 + P["w2"] * x2 + P["b1"])
    h2 = sigmoid(P["w3"] * x1 + P["w4"] * x2 + P["b2"])
    y_hat = sigmoid(P["w5"] * h1 + P["w6"] * h2 + P["b3"])
    return h1, h2, y_hat

def backward(P, x1, x2, y):
    h1, h2, y_hat = forward(P, x1, x2)
    d3 = y_hat - y                         # ① 출력의 오차(9장)
    d1 = d3 * P["w5"] * h1 * (1 - h1)      # ③④ h1 쪽
    d2 = d3 * P["w6"] * h2 * (1 - h2)      # ③④ h2 쪽
    return {"w5": d3 * h1, "w6": d3 * h2, "b3": d3,          # ②
            "w1": d1 * x1, "w2": d1 * x2, "b1": d1,          # ⑤
            "w3": d2 * x1, "w4": d2 * x2, "b2": d2}

h1, h2, y_hat = forward(P, 1, 0)
print(f"순전파: h1 = {h1:.3f}, h2 = {h2:.3f}, y_hat = {y_hat:.3f}")
G = backward(P, 1, 0, 1)
for k in ["w5", "w6", "b3", "w1", "w2", "b1", "w3", "w4", "b2"]:
    print(f"dL/d{k} = {G[k] + 0:7.3f}")   # + 0은 -0.000을 0.000으로

파라미터를 하나씩 조금 키우고 줄여 손실의 차이로 기울기를 어림하는 수치 미분으로 검산합니다.

In [ ]:
def loss(P, x1, x2, y):
    y_hat = forward(P, x1, x2)[2]
    # 이진 교차 엔트로피(9장)
    return -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

h = 1e-5
for k in P:
    Pp, Pm = dict(P), dict(P)
    Pp[k] += h              # 이 파라미터만 조금 키우고
    Pm[k] -= h              # 조금 줄여서
    num = (loss(Pp, 1, 0, 1) - loss(Pm, 1, 0, 1)) / (2 * h)
    print(f"{k}: 역전파 {G[k] + 0:9.6f}, 수치 미분 {num + 0:9.6f}")

## 4. XOR 학습하기
9개 파라미터를 무작위로 정하고, 네 입력의 평균 기울기로 학습률 1의 경사 하강법을 5,000 에포크 진행합니다.

In [ ]:
X = [(0, 0), (0, 1), (1, 0), (1, 1)]
Y = [0, 1, 1, 0]
rng = np.random.default_rng(0)
names = ["w1", "w2", "b1", "w3", "w4", "b2", "w5", "w6", "b3"]
P = {k: rng.normal() for k in names}       # 무작위로 처음 값 정하기
lr = 1.0
for epoch in range(1, 5001):
    G_sum = {k: 0.0 for k in P}
    for (x1, x2), y in zip(X, Y):      # 네 입력의 기울기를 더해
        G = backward(P, x1, x2, y)
        for k in P:
            G_sum[k] += G[k]
    for k in P:                        # 평균 기울기로 한 걸음
        P[k] -= lr * G_sum[k] / 4
    if epoch in (1, 100, 1000, 5000):
        L = np.mean([loss(P, x1, x2, y)
                     for (x1, x2), y in zip(X, Y)])
        print(f"에포크 {epoch:4d}: 평균 손실 {L:.4f}")
for x1, x2 in X:
    h1, h2, y_hat = forward(P, x1, x2)
    print(f"입력 ({x1}, {x2}) → h1 {h1:.2f}, h2 {h2:.2f}, "
          f"y_hat {y_hat:.3f}")

손실의 변화와 학습된 은닉 평면을 그려 봅니다(이북 그림 11.6). 처음 수백 에포크는 손실이 0.693 근처에서 평평합니다.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)                 # 위와 같은 처음 값으로 다시 학습하며 손실을 모두 기록
P = {k: rng.normal() for k in names}
hist = []
for epoch in range(5000):
    G_sum = {k: 0.0 for k in P}
    for (x1, x2), y in zip(X, Y):
        G = backward(P, x1, x2, y)
        for k in P:
            G_sum[k] += G[k]
    for k in P:
        P[k] -= 1.0 * G_sum[k] / 4
    hist.append(np.mean([loss(P, x1, x2, y) for (x1, x2), y in zip(X, Y)]))

H = np.array([forward(P, x1, x2)[:2] for x1, x2 in X])
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].semilogy(range(1, 5001), hist); axs[0].set_xlabel("epoch"); axs[0].set_ylabel("mean loss")
axs[1].scatter(H[:, 0], H[:, 1], c=["tab:blue", "tab:red", "tab:red", "tab:blue"], s=80)
for (x1, x2), (a, b) in zip(X, H):
    axs[1].annotate(f"({x1},{x2})", (a, b), textcoords="offset points", xytext=(6, 6))
axs[1].set_xlabel("h1"); axs[1].set_ylabel("h2"); axs[1].set_xlim(-0.2, 1.3); axs[1].set_ylim(-0.2, 1.3)
plt.tight_layout(); plt.show()

## 5. 자동 미분
텐서플로의 `GradientTape`는 순전파를 기록해 두었다가 거꾸로 따라가며 기울기를 구합니다. 먼저 2절의 선형 회귀 예입니다.

In [ ]:
import tensorflow as tf

x, y = 2.0, 7.0
w = tf.Variable(1.0)                # 학습할 파라미터
b = tf.Variable(1.0)
with tf.GradientTape() as tape:     # 이 안의 계산을 기록
    y_hat = w * x + b
    L = (y_hat - y) ** 2
dw, db = tape.gradient(L, [w, b])   # 기록을 거꾸로 따라가기
print("손실:", L.numpy())
print("dL/dw:", dw.numpy(), " dL/db:", db.numpy())

2-2-1 신경망의 기울기 9개도 자동 미분으로 구해 3절의 넘파이 역전파(이북 표 11.1)와 비교합니다.

In [ ]:
P0 = {"w1": 0.6, "w2": 0.4, "b1": 0.5, "w3": -0.4, "w4": 0.3, "b2": -0.7,
      "w5": 1.0, "w6": -1.0, "b3": -0.5}      # 11.4절의 처음 값
V = {k: tf.Variable(v) for k, v in P0.items()}
x1, x2, y = 1.0, 0.0, 1.0
with tf.GradientTape() as tape:
    h1 = tf.sigmoid(V["w1"] * x1 + V["w2"] * x2 + V["b1"])
    h2 = tf.sigmoid(V["w3"] * x1 + V["w4"] * x2 + V["b2"])
    y_hat = tf.sigmoid(V["w5"] * h1 + V["w6"] * h2 + V["b3"])
    L = -(y * tf.math.log(y_hat) + (1 - y) * tf.math.log(1 - y_hat))
grads = tape.gradient(L, list(V.values()))
G0 = backward(P0, 1, 0, 1)                      # 넘파이로 짠 역전파
for k, g in zip(V, grads):
    print(f"{k}: 자동 미분 {g.numpy() + 0:9.6f}, 넘파이 역전파 {G0[k] + 0:9.6f}")

## 6. 기울기 소실
은닉층 10개(층마다 뉴런 10개)인 신경망에 반달 데이터(6장, 9장)를 넣고 역전파로 층별 가중치 기울기의 평균 크기를 잽니다(이북 그림 11.7 ②). 한 층의 뉴런을 한꺼번에 계산하려고 행렬 곱셈(`@`)을 씁니다. 이북 11.4절 심화의 식을 한 층씩 행렬 곱셈으로 계산한 것입니다.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

Xm, ym = make_moons(n_samples=200, noise=0.2, random_state=0)
Xm = StandardScaler().fit_transform(Xm)
ym = ym.reshape(-1, 1)

def layer_grads(act, depth=10, width=10, seed=0):
    """은닉층 depth개(층마다 뉴런 width개)인 신경망을 순전파·역전파하고 층별 |dL/dW|의 평균을 돌려준다."""
    rng = np.random.default_rng(seed)
    sizes = [2] + [width] * depth + [1]
    scale = 2.0 if act == "relu" else 1.0       # 활성화 함수에 맞춘 처음 가중치 범위
    W = [rng.normal(0, np.sqrt(scale / sizes[i]), (sizes[i], sizes[i + 1])) for i in range(len(sizes) - 1)]
    b = [np.zeros(s) for s in sizes[1:]]
    A, Z = [Xm], []                             # 순전파: 층마다 출력(A)과 가중합(Z)을 저장
    for i in range(len(W)):
        z = A[-1] @ W[i] + b[i]                 # 행렬 곱셈으로 한 층의 가중합을 한꺼번에
        Z.append(z)
        last = (i == len(W) - 1)
        A.append(sigmoid(z) if (last or act == "sigmoid") else np.maximum(0, z))
    d = (A[-1] - ym) / len(Xm)                  # ① 출력의 오차(평균)
    g = []
    for i in reversed(range(len(W))):           # 역전파: 출력 쪽 층부터
        g.append(np.abs(A[i].T @ d).mean())     # 이 층 가중치의 기울기 크기
        if i > 0:
            d = d @ W[i].T                      # 앞 층 출력으로 전하기
            d = d * (A[i] * (1 - A[i]) if act == "sigmoid" else (Z[i - 1] > 0))   # 도함수 곱하기
    return g[::-1]                              # 입력 쪽 층부터 차례로

gs, gr = layer_grads("sigmoid"), layer_grads("relu")
for i, (a, c) in enumerate(zip(gs, gr), start=1):
    print(f"층 {i:2d}: 시그모이드 {a:.1e}, ReLU {c:.1e}")
print("0.25 ** 10 =", 0.25 ** 10)
plt.semilogy(range(1, 12), gs, "o-", label="sigmoid")
plt.semilogy(range(1, 12), gr, "s--", label="ReLU")
plt.xlabel("layer (1 = input side)"); plt.ylabel("mean |dL/dW|"); plt.legend(); plt.show()

---
## 실습 문제

**문제 1.** 이북 그림 11.5의 신경망(처음 값 `P0`)에 입력 $(0, 1)$, 정답 1을 넣고 `forward()`와 `backward()`로 순전파 값과 기울기 9개를 구하세요. 기울기가 0인 가중치는 무엇이고, 왜 0인가요? 학습률 1로 한 번 고친 뒤 $\hat{y}$와 손실은 어떻게 변하나요?

In [ ]:
# 문제 1 풀이

**문제 2.** 처음 값 `P0`, 입력 (1, 0), 정답 1에서 $w_5$의 기울기를 수치 미분으로 구합니다. $h$를 0.1, 0.01, 0.001로 바꾸며 한쪽으로만 바꾸는 방법 $(L(w + h) - L(w))/h$와 양쪽으로 바꾸는 방법 $(L(w + h) - L(w - h))/2h$의 값을 출력하고, 역전파로 구한 값과의 차이를 비교하세요.

In [ ]:
# 문제 2 풀이

**문제 3.** XOR 학습에서 학습률을 0.5, 1, 2, 10으로 바꾸어(처음 값은 `default_rng(0)`으로 같게) 평균 손실이 처음으로 0.1 아래로 내려가는 에포크를 구하세요. 학습률을 키우면 늘 좋을까요? 8장에서 배운 내용과 함께 생각해 보세요.

In [ ]:
# 문제 3 풀이

**문제 4.** 6절의 `layer_grads()`로 은닉층 수를 5, 10, 20개로 바꾸며 시그모이드와 ReLU 각각 (첫 층의 기울기 크기) ÷ (마지막 층의 기울기 크기)를 출력하세요. 층이 깊어질수록 두 활성화 함수의 차이는 어떻게 되나요?

In [ ]:
# 문제 4 풀이